# Phase 22 — Final Feature Leakage Audit

**Project:** Australian Stock Market Analytics & Return Modeling

## Objective
Create a provenance/timing audit, exclude target-construction and future-information fields from predictors, and compare a baseline with two models using the fixed chronological train/validation periods only.

**Safeguards:** The notebook does not score the test set. An automated name allowlist is only a first-pass screen, not proof that a feature is point-in-time safe. Review feature calculations and benchmark alignment manually before final evaluation.

## 1. Setup and locate project files

In [1]:
from pathlib import Path
import json, re
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
EXPANDED_REL = Path("data/processed/expanded_features/asx_3m_expanded_features_targets.csv")
TRAIN_REL = Path("data/processed/model_features/asx_3m_strict_split_train.csv")
VALID_REL = Path("data/processed/model_features/asx_3m_strict_split_validation.csv")
TEST_REL = Path("data/processed/model_features/asx_3m_strict_split_test.csv")

def locate_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if all((p / r).exists() for r in [EXPANDED_REL, TRAIN_REL, VALID_REL, TEST_REL]):
            return p
    for p in start.iterdir() if start.exists() else []:
        if p.is_dir() and all((p / r).exists() for r in [EXPANDED_REL, TRAIN_REL, VALID_REL, TEST_REL]):
            return p
    return None

PROJECT_ROOT = locate_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate Phase 19 expanded table and Phase 13 split files.")
OUT = PROJECT_ROOT / "reports" / "final_leakage_audit"
OUT.mkdir(parents=True, exist_ok=True)
CLEAN_OUT = PROJECT_ROOT / "data" / "processed" / "leakage_audited"
CLEAN_OUT.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)

Project root: e:\Projects\Australian Stock Market Data Science Project


## 2. Load source tables and define fixed time boundaries

In [2]:
def read_table(path):
    d = pd.read_csv(path, low_memory=False)
    d.columns = [str(c).strip().lower() for c in d.columns]
    for c in ["date", "target_endpoint_date_3m"]:
        if c in d.columns:
            d[c] = pd.to_datetime(d[c], errors="coerce")
    if "ticker" in d:
        d["ticker"] = d["ticker"].astype(str).str.strip().str.upper()
    return d

expanded = read_table(PROJECT_ROOT / EXPANDED_REL)
train_source = read_table(PROJECT_ROOT / TRAIN_REL)
valid_source = read_table(PROJECT_ROOT / VALID_REL)
test_source = read_table(PROJECT_ROOT / TEST_REL)
required = {"ticker", "date", "target_endpoint_date_3m", "forward_return_3m"}
for name, table in [("expanded", expanded), ("train", train_source), ("validation", valid_source), ("test", test_source)]:
    missing = required - set(table.columns)
    if missing:
        raise ValueError(f"{name} table missing columns: {sorted(missing)}")
train_start, train_end = train_source["date"].min(), train_source["date"].max()
valid_start, valid_end = valid_source["date"].min(), valid_source["date"].max()
test_start = test_source["date"].min()
print("Train period:", train_start, "to", train_end)
print("Validation period:", valid_start, "to", valid_end)
print("Test start:", test_start)

Train period: 2021-10-19 00:00:00 to 2024-11-14 00:00:00
Validation period: 2025-02-18 00:00:00 to 2025-08-05 00:00:00
Test start: 2025-11-03 00:00:00


## 3. Create a conservative provenance/timing screen

In [3]:
metadata_cols = {
    "ticker", "date", "target_endpoint_date_3m", "forward_return_3m",
    "target_endpoint_price_3m", "close_for_target", "close_for_features",
    "target_horizon_observations", "target_source_price_column", "target_definition"
}
unsafe_name_pattern = re.compile(
    r"(forward_return|target_endpoint|target_price|future|next_close|label|target_horizon|target_definition|close_for_target|close_for_features)",
    flags=re.IGNORECASE
)
numeric_cols = [c for c in expanded.columns if pd.api.types.is_numeric_dtype(expanded[c])]
candidate_numeric = [c for c in numeric_cols if c not in metadata_cols]
allow_pattern = re.compile(
    r"(momentum_(1|5|21|63|126)d|volatility_(5|21|63)d|price_to_sma_(21|63|126)d|short_term_reversal_1d|benchmark_return_(1|5|21|63|126)d|benchmark_volatility_(5|21|63)d|relative_momentum_(5|21|63|126)d)"
)
audit_rows = []
for col in candidate_numeric:
    reasons = []
    if unsafe_name_pattern.search(col):
        reasons.append("name indicates target/future/raw target-price field")
    if col.endswith(("_x", "_y")):
        reasons.append("merge-suffixed column; provenance requires review")
    allowed = bool(allow_pattern.fullmatch(col))
    if not allowed:
        reasons.append("not on Phase 19 engineered-feature allowlist")
    audit_rows.append({
        "feature": col, "allowed_by_name": allowed,
        "excluded_from_model": bool(reasons),
        "audit_reasons": "; ".join(reasons) if reasons else "candidate for calculation/timing review",
        "missing_rate": float(expanded[col].isna().mean()),
        "unique_values": int(expanded[col].nunique(dropna=True))
    })
audit = pd.DataFrame(audit_rows)
display(audit.sort_values(["excluded_from_model", "feature"], ascending=[False, True]))
audit.to_csv(OUT / "feature_provenance_timing_audit.csv", index=False)
safe_candidates = audit.loc[audit["allowed_by_name"] & ~audit["excluded_from_model"], "feature"].tolist()
print("Safe-by-name candidates (still require manual provenance review):", safe_candidates)

,feature,allowed_by_name,excluded_from_model,audit_reasons,missing_rate,unique_values
2,benchmark_return_1d_x,False,True,merge-suffixed column; provenance requires rev...,0.000912,1075
17,benchmark_return_1d_y,False,True,merge-suffixed column; provenance requires rev...,0.000912,1075
3,benchmark_volatility_21d_x,False,True,merge-suffixed column; provenance requires rev...,0.020289,1054
23,benchmark_volatility_21d_y,False,True,merge-suffixed column; provenance requires rev...,0.019359,1055
4,benchmark_volatility_63d_x,False,True,merge-suffixed column; provenance requires rev...,0.059328,1012
24,benchmark_volatility_63d_y,False,True,merge-suffixed column; provenance requires rev...,0.058399,1013
0,momentum_21d_x,False,True,merge-suffixed column; provenance requires rev...,0.020449,51582
7,momentum_21d_y,False,True,merge-suffixed column; provenance requires rev...,0.019520,51625
1,momentum_63d_x,False,True,merge-suffixed column; provenance requires rev...,0.059489,50058
8,momentum_63d_y,False,True,merge-suffixed column; provenance requires rev...,0.058560,50104


Safe-by-name candidates (still require manual provenance review): ['momentum_1d', 'momentum_5d', 'momentum_126d', 'volatility_5d', 'volatility_21d', 'volatility_63d', 'price_to_sma_21d', 'price_to_sma_63d', 'price_to_sma_126d', 'short_term_reversal_1d', 'benchmark_return_5d', 'benchmark_return_21d', 'benchmark_return_63d', 'benchmark_return_126d', 'benchmark_volatility_5d', 'relative_momentum_5d', 'relative_momentum_21d', 'relative_momentum_63d', 'relative_momentum_126d']


## 4. Compare suffixed duplicate variants against canonical values

In [4]:
suffix_report_rows = []
for col in expanded.columns:
    if col.endswith(("_x", "_y")):
        base = col[:-2]
        other = base + ("_y" if col.endswith("_x") else "_x")
        if other in expanded.columns:
            a, b = expanded[col], expanded[other]
            overlap = a.notna() & b.notna()
            suffix_report_rows.append({
                "base": base, "column_a": col, "column_b": other,
                "overlap_rows": int(overlap.sum()),
                "identical_full_series": bool(a.equals(b)),
                "equal_fraction_on_overlap": float((a[overlap].astype(str).values == b[overlap].astype(str).values).mean()) if overlap.any() else np.nan
            })
suffix_report = pd.DataFrame(suffix_report_rows).drop_duplicates(subset=["base"]) if suffix_report_rows else pd.DataFrame()
display(suffix_report)
suffix_report.to_csv(OUT / "suffix_variant_audit.csv", index=False)

,base,column_a,column_b,overlap_rows,identical_full_series,equal_fraction_on_overlap
0,momentum_21d,momentum_21d_x,momentum_21d_y,54799,False,0.016971
1,momentum_63d,momentum_63d_x,momentum_63d_y,52615,False,0.016212
2,benchmark_return_1d,benchmark_return_1d_x,benchmark_return_1d_y,55892,True,1.000000
3,benchmark_volatility_21d,benchmark_volatility_21d_x,benchmark_volatility_21d_y,54808,False,0.000000
4,benchmark_volatility_63d,benchmark_volatility_63d_x,benchmark_volatility_63d_y,52624,False,0.000000


## 5. Reconstruct fixed train/validation rows and verify endpoints

In [5]:
train = expanded[
    expanded["date"].between(train_start, train_end) &
    (expanded["target_endpoint_date_3m"] < valid_start)
].copy()
valid = expanded[
    expanded["date"].between(valid_start, valid_end) &
    (expanded["target_endpoint_date_3m"] < test_start)
].copy()
if train.empty or valid.empty:
    raise ValueError("Train or validation rows are empty after chronological/endpoint filtering.")
assert train["target_endpoint_date_3m"].max() < valid_start
assert valid["target_endpoint_date_3m"].max() < test_start
safe_features = [
    c for c in safe_candidates if c in train.columns and c in valid.columns
    and pd.to_numeric(train[c], errors="coerce").notna().any()
    and pd.to_numeric(train[c], errors="coerce").nunique(dropna=True) > 1
]
print("Train rows:", len(train), "| Validation rows:", len(valid))
print("Safe candidate feature count:", len(safe_features))
print(safe_features)

Train rows: 40447 | Validation rows: 6084
Safe candidate feature count: 19
['momentum_1d', 'momentum_5d', 'momentum_126d', 'volatility_5d', 'volatility_21d', 'volatility_63d', 'price_to_sma_21d', 'price_to_sma_63d', 'price_to_sma_126d', 'short_term_reversal_1d', 'benchmark_return_5d', 'benchmark_return_21d', 'benchmark_return_63d', 'benchmark_return_126d', 'benchmark_volatility_5d', 'relative_momentum_5d', 'relative_momentum_21d', 'relative_momentum_63d', 'relative_momentum_126d']


## 6. Train validation-only model comparison

In [6]:
X_train = train[safe_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
X_valid = valid[safe_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
y_train = pd.to_numeric(train["forward_return_3m"], errors="coerce")
y_valid = pd.to_numeric(valid["forward_return_3m"], errors="coerce")
train_mask, valid_mask = y_train.notna(), y_valid.notna()
X_train, y_train = X_train.loc[train_mask], y_train.loc[train_mask]
X_valid, y_valid = X_valid.loc[valid_mask], y_valid.loc[valid_mask]
if X_train.empty or X_valid.empty:
    raise ValueError("No valid rows available for model comparison.")
models = {
    "Training mean baseline": Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", DummyRegressor(strategy="mean"))]),
    "Leakage-audited Ridge": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", Ridge(alpha=10.0))]),
    "Leakage-audited Random Forest": Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", RandomForestRegressor(n_estimators=250, min_samples_leaf=25, max_features=0.8, random_state=RANDOM_STATE, n_jobs=-1))])
}
rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    prediction = model.predict(X_valid)
    rows.append({
        "model": name, "n_features": len(safe_features), "n_validation": len(y_valid),
        "MAE": mean_absolute_error(y_valid, prediction),
        "RMSE": np.sqrt(mean_squared_error(y_valid, prediction)),
        "R2": r2_score(y_valid, prediction),
        "actual_mean": float(y_valid.mean()), "predicted_mean": float(np.mean(prediction)),
        "mean_prediction_bias": float(np.mean(prediction) - y_valid.mean())
    })
results = pd.DataFrame(rows).sort_values("MAE").reset_index(drop=True)
display(results)
results.to_csv(OUT / "leakage_audited_validation_comparison.csv", index=False)

,model,n_features,n_validation,MAE,RMSE,R2,actual_mean,predicted_mean,mean_prediction_bias
0,Leakage-audited Ridge,19,6084,0.141023,0.195802,-0.089408,0.075098,0.029378,-0.045720
1,Training mean baseline,19,6084,0.141984,0.195113,-0.081749,0.075098,0.021461,-0.053637
2,Leakage-audited Random Forest,19,6084,0.148972,0.209516,-0.247350,0.075098,0.032391,-0.042707


## 7. Save audited feature table and summary

In [7]:
keep_cols = list(dict.fromkeys(
    [c for c in metadata_cols if c in expanded.columns] +
    ["ticker", "date", "target_endpoint_date_3m", "forward_return_3m"] + safe_features
))
audited_table = expanded[keep_cols].copy()
audited_table.to_csv(CLEAN_OUT / "asx_3m_leakage_audited_features_targets.csv", index=False)
baseline_row = results[results["model"] == "Training mean baseline"].iloc[0]
best_row = results.iloc[0]
summary = {
    "phase": 22, "expanded_rows": int(len(expanded)),
    "candidate_numeric_columns": int(len(candidate_numeric)),
    "safe_by_name_feature_count": int(len(safe_candidates)),
    "usable_model_feature_count": int(len(safe_features)),
    "model_features": safe_features, "train_rows": int(len(X_train)),
    "validation_rows": int(len(X_valid)), "best_validation_model": str(best_row["model"]),
    "best_validation_mae": float(best_row["MAE"]),
    "baseline_validation_mae": float(baseline_row["MAE"]),
    "best_beats_baseline": bool(best_row["MAE"] < baseline_row["MAE"]),
    "test_rows_scored": 0, "manual_provenance_review_required": True,
    "warning": "Automated feature-name allowlist is not proof of point-in-time safety. Verify feature calculations and benchmark alignment before final test evaluation."
}
with open(OUT / "phase22_final_leakage_audit_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, default=str)
print(json.dumps(summary, indent=2, default=str))
print("\nReport folder:", OUT)
print("Audited table:", CLEAN_OUT / "asx_3m_leakage_audited_features_targets.csv")

{
  "phase": 22,
  "expanded_rows": 55943,
  "candidate_numeric_columns": 29,
  "safe_by_name_feature_count": 19,
  "usable_model_feature_count": 19,
  "model_features": [
    "momentum_1d",
    "momentum_5d",
    "momentum_126d",
    "volatility_5d",
    "volatility_21d",
    "volatility_63d",
    "price_to_sma_21d",
    "price_to_sma_63d",
    "price_to_sma_126d",
    "short_term_reversal_1d",
    "benchmark_return_5d",
    "benchmark_return_21d",
    "benchmark_return_63d",
    "benchmark_return_126d",
    "benchmark_volatility_5d",
    "relative_momentum_5d",
    "relative_momentum_21d",
    "relative_momentum_63d",
    "relative_momentum_126d"
  ],
  "train_rows": 40447,
  "validation_rows": 6084,
  "best_validation_model": "Leakage-audited Ridge",
  "best_validation_mae": 0.14102338062093311,
  "baseline_validation_mae": 0.14198443955103976,
  "best_beats_baseline": true,
  "test_rows_scored": 0,
  "manual_provenance_review_required": true,
  "warning": "Automated feature-name al

## 8. Completion checklist

- Review `feature_provenance_timing_audit.csv`.
- Review suffix-pair comparisons and verify source calculations.
- Confirm `close_for_target` and target endpoint fields are excluded from model predictors.
- Verify benchmark features use information available by each prediction date.
- Review validation metrics against the baseline.
- Keep test metrics untouched until feature definitions are finalized.